# 25 · Apply ruleset over a shape, checked against Shapely

## What this checks

Can a rule judge a number worked out from a shape, not one somebody typed into a table? And does the judgement come
back attached to the shape it is about?

Two widgets make one chain here, both at release 0.1.0. The extent review rule that Apply ruleset ran here is now
retired, so this notebook is the record of it:

- **Convex hull** draws the tightest outline around a set of points and measures its area.
- **Apply ruleset** then classifies that area against a threshold.

The data are 250 located cholera deaths. The area is measured in UTM zone 30 north (EPSG:32630), a flat grid in
metres. The threshold is 250,000 m².

Checking the rule alone would prove nothing, because the rule is just a comparison. The real doubt was whether a
number worked out from a shape could reach a rule at all. Until 14 October 2026 it could not. Fieldwork computed the
hull's area, wrote it in the run's record, and dropped it before anything downstream could read it. So we check the
whole chain. A wrong status here is as likely to be a wrong area as a wrong threshold.

## How it is checked

We rebuild every step from what Fieldwork declares, with **Shapely**, Python's geometry library, built on GEOS. The
area is projected with **pyproj**, the Python link to the PROJ projection library. No code is shared with Fieldwork.

## What should happen

1. **The outline, corner for corner.** A convex hull is exact, with no approximation, so any difference is real. We
   compare within 0.000001 degrees, about 0.1 mm.
2. **The area within 0.01%.** Both sides use the same shoelace formula over projected metres, so they should agree
   almost exactly. The margin is for the projection library, not the arithmetic.
3. **The same status, for the same stated reason.** The status is Unknown when no area is attached, Review above the
   threshold, and NoFlag at or below it. A status could agree by accident: the right word from the wrong numbers. So
   the explanation must carry the same two numbers too.

These criteria were fixed before the check was run.

In [ ]:
NEEDED = ["shapely", "pyproj", "numpy"]
try:
    import pyodide_js
    await pyodide_js.loadPackage(NEEDED)
    print("Requested from the Pyodide distribution:", ", ".join(NEEDED))
except ImportError:
    print("Not a Pyodide kernel; using the ambient environment.")
except Exception as error:
    print("loadPackage failed:", type(error).__name__, error)
    try:
        import piplite
        await piplite.install(NEEDED)
    except Exception as fallback:
        print("piplite also failed:", type(fallback).__name__, fallback)

In [ ]:
import json, sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd()))
from checks.check_25_extent import run

report = run(Path.cwd() / "fixtures" / "extent-soho-001.json")
print("criterion:", report["criterion"])
print("agrees:", report["agrees"])
print(f"points hulled: {report['points']}")
print(f"hull vertices: Fieldwork {report['fieldwork']['hullVertices']}, "
      f"Shapely {report['reference']['hullVertices']}, rings agree: {report['ringsAgree']}")
print(f"area: Fieldwork {report['fieldwork']['areaSquareMetres']:.1f} m2, "
      f"Shapely {report['reference']['areaSquareMetres']:.1f} m2 "
      f"({report['areaDifferenceFraction']*100:.4f}%)")
print(f"status: Fieldwork {report['fieldwork']['status']}, "
      f"Shapely {report['reference']['status']}")
print("because:", report["fieldwork"]["explanation"])

## The result on record

Recorded on 14 October 2026, against release 0.1.0 of both widgets: **Agrees**. On 26 November 2026 it was run again
on the same data, not on new data, because the rule it checks is retired and nothing can export its data now. It
still agrees.

Both sides drew the same 11-cornered outline, corner for corner. Both measured 139,230 m²; the difference rounds to
0.0000%. Both gave the status NoFlag, and the explanation carries the same two numbers.

Running the cell above repeats the check on the same data.

## What this does not show

- That 250,000 m² is the right threshold, or that a convex hull is the right outline. Both are the practitioner's to
  choose. Neither Fieldwork nor this notebook gives a basis for either.
- That the rule fired in the reasoner. The data record what the reasoner (EYE) concluded, and we redo the same
  comparison in Python. That checks the area and the threshold, not the reasoning. [Check
  23](./23-rulesets-vs-decision-table.ipynb) checks the rules themselves, against a full decision table.
- Anything about ground the hull does not cover. A hull outlines **what was observed**. It encloses ground where
  nothing was recorded, and says nothing about what lies outside it.